# DPO From Scratch

Companion notebook for the [Direct Preference Optimization](https://codemia.io/courses/llm_inside_and_out/direct_preference_optimization) lesson on Codemia.

**What you will build**: the DPO loss function from the Rafailov et al. paper, implemented by hand (no `trl` library). Train a small LoRA-wrapped distilgpt2 on a tiny preference dataset (chosen / rejected pairs). Compare the DPO-aligned model against an SFT baseline on held-out preference evaluation prompts.

**Time**: ~25-40 minutes on T4

**GPU required**: yes

## Setup

In [ ]:
%pip install -q torch transformers peft datasets matplotlib

In [ ]:
import copy, random, math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

torch.manual_seed(0)
random.seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## Concept recap

DPO (Rafailov et al., 2023) replaces the PPO-based RLHF pipeline with a single supervised-looking loss. Given a preference dataset of `(prompt, chosen, rejected)` triples and a reference model `pi_ref` (usually the SFT checkpoint), DPO trains `pi_theta` to maximize:

```
L = -log sigmoid( beta * [ log(pi_theta(y_w | x) / pi_ref(y_w | x))
                          - log(pi_theta(y_l | x) / pi_ref(y_l | x)) ] )
```

In English: increase the log-likelihood of the chosen response relative to the reference, and decrease the rejected response's log-likelihood relative to the reference. `beta` controls how much the policy is allowed to drift from the reference; larger `beta` means stricter.

Because the reference is frozen, DPO never explicitly maximizes a reward. It directly optimizes the preference-consistent policy that PPO would find if PPO were run to convergence. And because both branches use the same loss shape, it's just SFT-with-two-heads in implementation.

## Data: a hand-crafted preference dataset

In [ ]:
# Real DPO needs thousands of pairs; this is a tiny pedagogical set.
# Theme: reward concise, factual, polite answers; downweight rambling, rude, or wrong ones.
prefs = [
    ("What is the capital of France?",
     "The capital of France is Paris.",
     "uh idk maybe London? or Rome? hard to say really"),
    ("Explain photosynthesis briefly.",
     "Photosynthesis is the process by which plants use sunlight to convert carbon dioxide and water into glucose and oxygen.",
     "plants eat the sun and poop out air lol"),
    ("What is 2 + 2?",
     "2 + 2 equals 4.",
     "2 + 2 is somewhere around 5 I think"),
    ("Who wrote Romeo and Juliet?",
     "Romeo and Juliet was written by William Shakespeare.",
     "some old guy from England idk his name"),
    ("What is water's chemical formula?",
     "Water's chemical formula is H2O.",
     "water is just water obviously you don't need a formula"),
    ("How many continents are there?",
     "There are seven continents on Earth.",
     "there are like a bunch maybe 5 or 6 or 7 who counts"),
    ("What color is the sky on a clear day?",
     "The sky appears blue on a clear day due to Rayleigh scattering of sunlight.",
     "sky??? green??? purple??? pick one"),
    ("Name a famous physicist.",
     "Albert Einstein is a famous physicist best known for the theory of relativity.",
     "IDK MAN PHYSICS IS HARD STOP ASKING ME"),
    ("What is the boiling point of water at sea level?",
     "Water boils at 100 degrees Celsius (212 F) at sea level.",
     "water boils when its really hot like 9000 degrees"),
    ("Who painted the Mona Lisa?",
     "The Mona Lisa was painted by Leonardo da Vinci.",
     "picasso maybe? or van gogh? it's one of the italian guys"),
    ("What is the fastest land animal?",
     "The cheetah is the fastest land animal, reaching speeds up to 70 mph.",
     "a really fast dog maybe?? or a horse on roller skates"),
    ("What language is spoken in Brazil?",
     "Portuguese is the official language of Brazil.",
     "brazilian? or spanish? or like... jungle language?"),
]
print(f"{len(prefs)} preference pairs")

## Load model, build SFT baseline and LoRA policy

In [ ]:
model_name = 'distilgpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Reference model: the frozen SFT checkpoint. In a real DPO pipeline you would
# first run SFT, then use that as reference. Here we use the base distilgpt2
# as both the reference and the starting point for the policy, which is a
# common simplification for small demos.
ref_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad = False
print('reference model loaded and frozen')

In [ ]:
# Policy = LoRA-wrapped distilgpt2. Same architecture, trainable low-rank delta.
policy_base = AutoModelForCausalLM.from_pretrained(model_name).to(device)
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8, lora_alpha=16, lora_dropout=0.0,
    target_modules=["c_attn"], bias="none",
)
policy = get_peft_model(policy_base, lora_config)
policy.print_trainable_parameters()

## Implementation: per-sequence log-prob under a model

In [ ]:
def seq_logprob(model, input_ids, attention_mask, response_mask):
    # Returns the summed log-prob of the response tokens under the given model.
    # input_ids: (B, T)  full sequence (prompt + response)
    # response_mask: (B, T) boolean, True on response positions (the ones we score)
    # We shift by one so position t predicts token t+1 -- standard LM convention.
    out = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = out.logits[:, :-1, :]          # drop last position (nothing to predict)
    targets = input_ids[:, 1:]              # shift labels
    rmask = response_mask[:, 1:]             # align the mask with the shift

    # Log-softmax and gather the target token's logp for each position.
    logp_all = F.log_softmax(logits, dim=-1)
    token_logp = logp_all.gather(2, targets.unsqueeze(-1)).squeeze(-1)  # (B, T-1)
    # Zero out prompt + padding positions and sum per-sequence.
    token_logp = token_logp * rmask.float()
    return token_logp.sum(dim=-1)  # (B,)

## Implementation: the DPO loss

In [ ]:
def dpo_loss(policy, ref, batch, beta=0.1):
    # batch: dict with chosen_input_ids, chosen_attention_mask, chosen_response_mask,
    #        rejected_input_ids, rejected_attention_mask, rejected_response_mask
    logp_w_policy = seq_logprob(policy, batch['chosen_input_ids'], batch['chosen_attention_mask'], batch['chosen_response_mask'])
    logp_l_policy = seq_logprob(policy, batch['rejected_input_ids'], batch['rejected_attention_mask'], batch['rejected_response_mask'])

    with torch.no_grad():
        logp_w_ref = seq_logprob(ref, batch['chosen_input_ids'], batch['chosen_attention_mask'], batch['chosen_response_mask'])
        logp_l_ref = seq_logprob(ref, batch['rejected_input_ids'], batch['rejected_attention_mask'], batch['rejected_response_mask'])

    # Log-ratios: how much more (or less) does the policy prefer each response relative to the ref?
    logratio_w = logp_w_policy - logp_w_ref
    logratio_l = logp_l_policy - logp_l_ref

    # The DPO objective: we want logratio_w > logratio_l (policy prefers chosen over rejected
    # MORE than the reference did). Wrapped in -log sigmoid so it's a minimizable loss.
    logits = beta * (logratio_w - logratio_l)
    loss = -F.logsigmoid(logits).mean()

    # Implicit reward: how much is the policy actually preferring chosen over rejected?
    # Useful to log alongside the loss.
    with torch.no_grad():
        margin = (logratio_w - logratio_l).mean().item()
        # Accuracy: fraction of pairs where policy prefers chosen (positive margin).
        acc = ((logratio_w - logratio_l) > 0).float().mean().item()
    return loss, margin, acc

## Batch encoding

In [ ]:
def encode_pair(prompt, chosen, rejected, max_len=128):
    # Encode prompt+response for both. Return a per-token response mask so we can
    # compute log-probs over response tokens only.
    pt = tokenizer.encode(prompt, add_special_tokens=False)
    ct = tokenizer.encode(' ' + chosen + tokenizer.eos_token, add_special_tokens=False)
    rt = tokenizer.encode(' ' + rejected + tokenizer.eos_token, add_special_tokens=False)

    chosen_ids = (pt + ct)[:max_len]
    rejected_ids = (pt + rt)[:max_len]

    # Response mask: 1 for positions inside ct/rt, 0 for prompt positions.
    c_mask = [0] * min(len(pt), len(chosen_ids)) + [1] * max(0, len(chosen_ids) - len(pt))
    r_mask = [0] * min(len(pt), len(rejected_ids)) + [1] * max(0, len(rejected_ids) - len(pt))
    return chosen_ids, c_mask, rejected_ids, r_mask

In [ ]:
def collate(pref_pairs):
    # Encode all pairs, then pad to the max length in the batch.
    encoded = [encode_pair(*p) for p in pref_pairs]
    max_c = max(len(e[0]) for e in encoded)
    max_r = max(len(e[2]) for e in encoded)
    max_len = max(max_c, max_r)
    pad = tokenizer.eos_token_id

    def pad_to(lst, L, val=pad):
        return lst + [val] * (L - len(lst))

    ci = torch.tensor([pad_to(e[0], max_len) for e in encoded], device=device)
    cm = torch.tensor([pad_to(e[1], max_len, 0) for e in encoded], device=device)
    ri = torch.tensor([pad_to(e[2], max_len) for e in encoded], device=device)
    rm = torch.tensor([pad_to(e[3], max_len, 0) for e in encoded], device=device)
    c_attn = torch.tensor([[1]*len(e[0]) + [0]*(max_len - len(e[0])) for e in encoded], device=device)
    r_attn = torch.tensor([[1]*len(e[2]) + [0]*(max_len - len(e[2])) for e in encoded], device=device)
    return dict(
        chosen_input_ids=ci, chosen_attention_mask=c_attn, chosen_response_mask=cm,
        rejected_input_ids=ri, rejected_attention_mask=r_attn, rejected_response_mask=rm,
    )

## Train loop

In [ ]:
# Hold out a couple pairs for eval
train_prefs = prefs[:-2]
eval_prefs = prefs[-2:]

opt = torch.optim.AdamW([p for p in policy.parameters() if p.requires_grad], lr=5e-4)

n_epochs = 40
batch_size = 4
beta = 0.1

history = []
for epoch in range(n_epochs):
    random.shuffle(train_prefs)
    epoch_loss = epoch_margin = epoch_acc = 0.0
    n_batches = 0
    for i in range(0, len(train_prefs), batch_size):
        batch = collate(train_prefs[i:i+batch_size])
        loss, margin, acc = dpo_loss(policy, ref_model, batch, beta=beta)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        epoch_loss += loss.item(); epoch_margin += margin; epoch_acc += acc; n_batches += 1
    history.append((
        epoch, epoch_loss/n_batches, epoch_margin/n_batches, epoch_acc/n_batches
    ))
    if epoch % 5 == 0 or epoch == n_epochs - 1:
        print(f"epoch {epoch:3d}  loss {epoch_loss/n_batches:.3f}  margin {epoch_margin/n_batches:+.3f}  train acc {epoch_acc/n_batches:.2f}")

## Curves

In [ ]:
ep, losses, margins, accs = zip(*history)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
axes[0].plot(ep, losses, 'o-'); axes[0].set_xlabel('epoch'); axes[0].set_ylabel('DPO loss'); axes[0].set_title('loss')
axes[1].plot(ep, margins, 'o-'); axes[1].set_xlabel('epoch'); axes[1].set_ylabel('margin'); axes[1].set_title('log-ratio margin (chosen - rejected)')
axes[2].plot(ep, accs, 'o-'); axes[2].set_xlabel('epoch'); axes[2].set_ylabel('pair acc'); axes[2].set_ylim(0, 1.05); axes[2].set_title('fraction preferred correct')
plt.tight_layout()
plt.show()

print("Loss drops, margin climbs, accuracy saturates near 1.0 -- the policy learned to prefer the chosen responses.")

## Evaluate on held-out preference pairs

For each held-out pair, score the chosen and rejected responses under both the reference and the trained policy. The policy should give the chosen response a higher relative log-prob than the reference does.

In [ ]:
@torch.no_grad()
def score_pair(model, prompt, resp):
    # Return the summed log-prob of the response tokens under the model.
    batch = collate([(prompt, resp, resp)])  # same response twice, only look at chosen
    lp = seq_logprob(model, batch['chosen_input_ids'], batch['chosen_attention_mask'], batch['chosen_response_mask'])
    return lp.item()

for prompt, chosen, rejected in eval_prefs:
    ref_c = score_pair(ref_model, prompt, chosen)
    ref_r = score_pair(ref_model, prompt, rejected)
    pol_c = score_pair(policy, prompt, chosen)
    pol_r = score_pair(policy, prompt, rejected)
    ref_margin = ref_c - ref_r
    pol_margin = pol_c - pol_r
    print(f"prompt: {prompt}")
    print(f"  reference margin (chosen-rejected): {ref_margin:+.2f}")
    print(f"  policy    margin (chosen-rejected): {pol_margin:+.2f}")
    print(f"  delta (how much DPO shifted preference): {pol_margin - ref_margin:+.2f}")
    print()

## Sample from each model

In [ ]:
@torch.no_grad()
def sample_reply(model, prompt, max_new=40):
    ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
    out = model.generate(ids, max_new_tokens=max_new, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

for prompt, _, _ in eval_prefs:
    print(f"prompt: {prompt}")
    print(f"  reference: {sample_reply(ref_model, prompt)}")
    print(f"  DPO model: {sample_reply(policy, prompt)}")
    print()

## Variation: beta sweep

`beta` controls how strongly DPO penalizes drift from the reference. Tiny beta lets the policy diverge freely (like pure supervised on chosen). Huge beta keeps it glued to the reference. Sweep and plot margin vs beta.

In [ ]:
def quick_dpo(beta, n_epochs=15):
    # Fresh LoRA policy every run so we isolate the effect of beta.
    pb = AutoModelForCausalLM.from_pretrained(model_name).to(device)
    p = get_peft_model(pb, lora_config)
    o = torch.optim.AdamW([q for q in p.parameters() if q.requires_grad], lr=5e-4)
    pairs = list(train_prefs)
    for _ in range(n_epochs):
        random.shuffle(pairs)
        for i in range(0, len(pairs), batch_size):
            b = collate(pairs[i:i+batch_size])
            l, _, _ = dpo_loss(p, ref_model, b, beta=beta)
            o.zero_grad(set_to_none=True); l.backward(); o.step()
    # Evaluate final margin on held-out.
    margins = []
    for prompt, chosen, rejected in eval_prefs:
        pc = score_pair(p, prompt, chosen)
        pr = score_pair(p, prompt, rejected)
        margins.append(pc - pr)
    return sum(margins) / len(margins)

betas = [0.01, 0.1, 0.5, 1.0]
margins_by_beta = []
for b in betas:
    m = quick_dpo(b)
    margins_by_beta.append(m)
    print(f"beta={b:<4}  held-out margin={m:+.3f}")

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(betas, margins_by_beta, 'o-')
ax.set_xscale('log')
ax.set_xlabel('beta')
ax.set_ylabel('held-out chosen-rejected margin')
ax.set_title('DPO margin vs beta')
plt.tight_layout()
plt.show()

## What to try next

- Swap in a real public preference dataset like `Anthropic/hh-rlhf` (subset) or `trl-lib/ultrafeedback_binarized`. The code above runs unchanged; just replace the `prefs` list.
- Implement **IPO** (Azar et al., 2023): replace `-log sigmoid(x)` with `(x - 1/(2*beta))^2`. It's a drop-in change to `dpo_loss`. Observe how it handles overconfident rewards.
- Compare with `trl.DPOTrainer` on the same data. Your losses should match within numerical precision.
- Track KL(policy || ref) as a separate metric and plot it over training. Large KL means the policy is drifting fast; small beta amplifies that.
- Try the **length-penalty** trick: some DPO implementations normalize log-probs by token count to prevent the model from preferring shorter responses purely on length.